# Exercice complet : l'électricité dans 12 quartiers

Une compagnie d'électricité (fictive) vous confie deux ans de relevés, de janvier 2024 à décembre 2025, pour 12 quartiers de Port-au-Prince, Delmas, Pétion-Ville et Cap-Haïtien. Le fichier arrive **brut**. Le directeur veut savoir :

> **La consommation suit-elle la chaleur, ou les heures de courant ?**

| Fichier | Une ligne par | Colonnes |
|---|---|---|
| `donnees/consommation_brut.csv` | quartier et mois | `mois`, `quartier`, `kwh` (consommation totale du quartier), `heures_courant` (heures de courant par jour, en moyenne) |
| `donnees/quartiers.csv` | quartier | `quartier`, `commune`, `type` (Résidentiel ou Commercial), `abonnes` |
| `donnees/meteo.csv` | commune et mois | `mois`, `commune`, `temperature_c`, `pluie_mm` |

Ce notebook suit **le même plan que celui du projet final** : diagnostic, nettoyage, jointures (qui écrivent le fichier propre), une partie par question, conclusion. Chaque étape donne le résultat **attendu** : vérifiez le vôtre avant de passer à la suite.

Deux livrables : **ce notebook**, exécuté du début à la fin (Restart, puis Run All), qui montre votre démarche, et **le tableau de bord `app.py`, en ligne**, qui montre vos résultats au directeur.

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, écritures différentes. On note chaque problème : on le corrigera à l'étape 2.

**1.** Lisez `consommation_brut.csv` dans `brut`. Donnez sa forme, le type de chaque colonne, le nombre de valeurs manquantes par colonne et le nombre de lignes en double.

Attendu : `(300, 4)` ; `kwh` est du texte (`str`) ; 9 manquants dans `kwh` ; 12 lignes en double.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

brut = pd.read_csv("donnees/consommation_brut.csv")
print(brut.shape)
print(brut.dtypes)
print(brut.isna().sum())
print(brut.duplicated().sum())

**2.** Combien d'écritures différentes pour la colonne `quartier` ? Il y a pourtant 12 quartiers.

Attendu : 43 écritures.

In [ ]:
print(brut["quartier"].nunique())

**3.** Pourquoi `kwh` n'est-il pas un nombre ? Affichez les valeurs que `pd.to_numeric(..., errors="coerce")` ne sait pas convertir (sans compter les vrais manquants).

Attendu : des valeurs comme « 184410 kWh » ou « 251 900 ».

In [ ]:
conversion = pd.to_numeric(brut["kwh"], errors="coerce")
print(brut.loc[conversion.isna() & brut["kwh"].notna(), "kwh"].unique())

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème du diagnostic, en comptant à chaque fois.

**Contrôle :** à la fin, `propre.shape` vaut `(279, 4)` et `propre` est identique au fichier de référence.

**4.** Dans une copie `propre`, convertissez `kwh` en nombre : retirez « kWh » et les espaces, puis `pd.to_numeric`. Combien reste-t-il de manquants ?

Attendu : 9 manquants, les vrais.

In [ ]:
propre = brut.copy()
propre["kwh"] = pd.to_numeric(
    propre["kwh"].str.replace("kWh", "", regex=False).str.replace(" ", "", regex=False),
    errors="coerce")
print(propre["kwh"].isna().sum())   # 9

**5.** Harmonisez les noms de quartiers. Attention : `.str.title()` transforme « Bas-Peu-de-Chose » en « Bas-Peu-De-Chose ». Utilisez plutôt les noms officiels de `quartiers.csv`, avec un dictionnaire qui part des noms en minuscules.

Attendu : 12 quartiers, aucun nom perdu par `map`.

In [ ]:
quartiers = pd.read_csv("donnees/quartiers.csv")
noms = {n.lower(): n for n in quartiers["quartier"]}
propre["quartier"] = propre["quartier"].str.strip().str.lower().map(noms)
print(propre["quartier"].isna().sum(), propre["quartier"].nunique())   # 0 12

**6.** Supprimez les doublons, puis les lignes sans `kwh` (avec `subset`). Combien de lignes reste-t-il ?

Attendu : `(279, 4)`.

In [ ]:
propre = propre.drop_duplicates().dropna(subset=["kwh"])
print(propre.shape)   # (279, 4)

**7.** Quatre relevés ont été saisis 10 fois trop grands. Trouvez-les avec la règle « plus de 3 fois la médiane de son quartier », divisez-les par 10, puis donnez le maximum de `kwh`.

Attendu : un relevé corrigé à Delmas 33, Delmas 75, Morne Calvaire et Rue 15 ; maximum 742780.

In [ ]:
mediane = propre.groupby("quartier")["kwh"].transform("median")
trop_grands = propre["kwh"] > 3 * mediane
print(propre[trop_grands])
propre.loc[trop_grands, "kwh"] = propre.loc[trop_grands, "kwh"] / 10
print(propre["kwh"].max())   # 742780.0

**8.** La preuve : comparez `propre` à `consommation.csv` (le fichier de référence), sur les mêmes lignes et les mêmes colonnes, avec `equals`.

Attendu : `True`.

In [ ]:
propre = propre.astype({"kwh": "int64"}).reset_index(drop=True)
reference = pd.read_csv("donnees/consommation.csv")
reference = reference.merge(propre[["mois", "quartier"]], on=["mois", "quartier"], how="right")[propre.columns]
print(propre.equals(reference))   # True

## 3. Jointures

On ajoute les informations des autres tables, en comptant les lignes avant et après chaque `merge`. Puis on écrit le fichier propre, celui que lira le tableau de bord.

**Contrôle :** 279 lignes du début à la fin.

**9.** Joignez `propre` et `quartiers`, puis ajoutez la colonne `kwh_par_abonne`, arrondie à 1 décimale, et la colonne `annee`.

Attendu : 279 lignes avant et après ; Bas-Peu-de-Chose en 2024-01 : 85.8 kWh par abonné.

In [ ]:
joint = propre.merge(quartiers, on="quartier", how="left")
joint["kwh_par_abonne"] = (joint["kwh"] / joint["abonnes"]).round(1)
joint["annee"] = joint["mois"].str[:4].astype(int)
print(len(propre), len(joint))   # 279 279

**10.** Joignez la météo : la clé est double, `mois` **et** `commune`. Vérifiez le nombre de lignes et les températures manquantes.

Attendu : 279 lignes avant et après, aucune température manquante.

In [ ]:
meteo = pd.read_csv("donnees/meteo.csv")
avec_meteo = joint.merge(meteo, on=["mois", "commune"], how="left")
print(len(avec_meteo), avec_meteo["temperature_c"].isna().sum())   # 279 0

**11.** Enregistrez `avec_meteo` dans `donnees/consommation_propre.csv`, sans index : c'est le fichier que lira le tableau de bord. Relisez-le et donnez sa forme.

Attendu : `(279, 11)`.

In [ ]:
avec_meteo.to_csv("donnees/consommation_propre.csv", index=False)
df = pd.read_csv("donnees/consommation_propre.csv")
print(df.shape)   # (279, 11)

## 4. Quels quartiers consomment le plus par abonné ?

**12.** Consommation moyenne par abonné selon le type de quartier, puis le classement des 12 quartiers.

Attendu : Commercial 393.2, Résidentiel 117.4 ; en tête : Centre-ville (468.7), Place Saint-Pierre (458.3), Delmas 33 (374.4).

In [ ]:
print(df.groupby("type")["kwh_par_abonne"].mean().round(1))
print(df.groupby("quartier")["kwh_par_abonne"].mean().round(1).sort_values(ascending=False))

**Ce qu'on en conclut :**

Les quartiers commerciaux consomment en moyenne 393.2 kWh par abonné, contre 117.4 pour les résidentiels, soit plus de 3 fois plus. Les trois premiers (Centre-ville 468.7, Place Saint-Pierre 458.3, Delmas 33 374.4) sont tous commerciaux.

## 5. Comment la consommation évolue-t-elle, commune par commune ?

**13.** Tableau croisé : consommation totale par commune et par année, puis la hausse en % (calculée avant d'arrondir), puis les totaux en millions de kWh arrondis à 2. Puis la courbe de la consommation par abonné, mois par mois, pour chaque commune.

Attendu : Cap-Haïtien en tête, de 4.59 à 5.62 millions de kWh (+22.4 %) ; Delmas en dernier (+1.1 %) ; 4 courbes.

In [ ]:
t = df.pivot_table(index="commune", columns="annee", values="kwh", aggfunc="sum")
t["hausse_pct"] = ((t[2025] / t[2024] - 1) * 100).round(1)
t[[2024, 2025]] = (t[[2024, 2025]] / 1e6).round(2)
print(t.sort_values("hausse_pct", ascending=False))

df.pivot_table(index="mois", columns="commune", values="kwh_par_abonne").plot(figsize=(10, 4))
plt.show()

**Ce qu'on en conclut :**

La consommation augmente partout, mais très inégalement. Cap-Haïtien est en tête (de 4.59 à 5.62 millions de kWh, environ +22 %), suivie de Pétion-Ville (+12.6 %) et de Port-au-Prince (+2.3 %). Delmas est presque stable (+1.1 %).

## 6. La consommation suit-elle la chaleur, ou les heures de courant ?

**14.** Quartiers résidentiels seulement : nuage de points des heures de courant contre la consommation par abonné. Puis la corrélation de la consommation par abonné avec `heures_courant`, et avec `temperature_c`.

Attendu : 0.76 avec les heures de courant, -0.15 avec la température.

In [ ]:
res = df[df["type"] == "Résidentiel"]
res.plot.scatter(x="heures_courant", y="kwh_par_abonne")
plt.show()
print(res["kwh_par_abonne"].corr(res["heures_courant"]).round(2))   # 0.76
print(res["kwh_par_abonne"].corr(res["temperature_c"]).round(2))    # -0.15

**Ce qu'on en conclut :**

Dans les quartiers résidentiels, la consommation par abonné est fortement liée aux heures de courant (corrélation 0.76). Elle l'est presque pas à la température (-0.15, et même légèrement négative). Les abonnés consomment donc davantage quand ils ont du courant, pas quand il fait chaud.

## 7. Où les coupures sont-elles les plus longues ?

**15.** Les heures de courant : médiane, Q1, Q3 par commune, puis un histogramme pour Cap-Haïtien. Combien de relevés de 2025 ont moins de 8 heures de courant par jour, et dans quels quartiers ?

Attendu : médiane de 10.0 heures à Cap-Haïtien, entre 11.6 et 11.9 ailleurs ; 3 relevés de 2025, à Carénage, Delmas 33 et Rue 15.

In [ ]:
s = df.groupby("commune")["heures_courant"].describe()[["25%", "50%", "75%"]]
print(s)
df[df["commune"] == "Cap-Haïtien"]["heures_courant"].plot.hist(bins=10)
plt.show()
print(df[(df["annee"] == 2025) & (df["heures_courant"] < 8)][["mois", "quartier", "heures_courant"]])

**Ce qu'on en conclut :**

Cap-Haïtien est la commune la moins bien desservie : médiane de 10.0 h de courant par jour, contre 11.6 à 11.9 h ailleurs. Seuls 3 relevés de 2025 passent sous 8 h : Delmas 33 (7.4 h), Carénage (7.3 h) et Rue 15 (5.9 h).

## 8. Quels relevés sortent du lot ?

**16.** Les 3 plus grosses consommations de tout le fichier : mois, quartier, type, kwh.

Attendu : Delmas 33 en 2025-07 (742780), Centre-ville en 2024-11 (597730), Delmas 33 en 2025-09 (591420).

In [ ]:
print(df.nlargest(3, "kwh")[["mois", "quartier", "type", "kwh"]])

**Ce qu'on en conclut :**

Les 3 plus grosses consommations viennent de quartiers commerciaux : Delmas 33 en 2025-07 (742 780 kWh) et en 2025-09 (591 420), et Centre-ville en 2024-11 (597 730). Ce sont de vraies valeurs élevées, pas des erreurs de saisie : celles-ci ont été corrigées à l'étape 7.

la consommation suit les heures de courant (corrélation 0.76), pas la chaleur (-0.15). Une corrélation ne prouve pas la cause : un autre facteur, comme le type de quartier, peut jouer.
Un seul chiffre diffère de l'énoncé : la hausse de Cap-Haïtien sort à +22.5 % au lieu de 22.4 %, ce qui est sans importance.

## 9. Conclusion

En trois phrases, répondez au directeur : la consommation suit-elle la chaleur, ou les heures de courant ? Citez vos chiffres, et dites ce qu'une corrélation ne permet pas de conclure.

La consommation suit les heures de courant, pas la chaleur : dans les quartiers résidentiels, la corrélation est de 0.76 avec les heures de courant, contre -0.15 avec la température. Plus il y a de courant, plus les abonnés consomment, ce qui est logique, car on ne peut pas consommer ce qu'on ne reçoit pas. Une corrélation ne prouve pas la cause : elle n'indique pas le sens de la relation et ne l'exclut pas qu'un autre facteur joue (type de quartier, équipements). Elle ne repose ici que sur 186 relevés résidentiels.

## Et maintenant : le tableau de bord

Le directeur ne lit pas de notebook : il ouvre un lien. La suite se fait dans `app.py`, en six étapes, avec `uv run streamlit run app.py` ouvert à côté. Les étapes sont dans le README.